# LCEL 심화 및 응용

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os

def project_setup(name):
    os.environ['LANGSMITH_PROJECT'] = name
    os.environ['LANGSMITH_TRACING'] = 'true'
    print('프로젝트: {} 추적 시작'.format(name))

def stop_tracing():
    os.environ['LANGSMITH_TRACING'] = 'false'
    print('LangSmith 추적 중지')

project_setup('LangChain Test')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "gpt-6-luna"

llm = ChatOpenAI(model=MODEL)
to_str = StrOutputParser()

## Step 01 · 파이프 합성 원리

- `a | b` → `RunnableSequence`
- 파이프 안의 dict는 `RunnableParallel`로,
- 함수는 → `RunnableLambda` 로 자동 변환

In [ ]:
prompt = ChatPromptTemplate.from_template("{topic}을(를) 한 문장으로 설명해줘.")
chain = prompt | llm | to_str
print(type(chain).__name__)
print(chain.steps)   # 체인을 구성하는 Runnable 목록

In [ ]:
# dict와 함수의 자동 변환 확인
auto = {"topic": lambda x: x.upper()} | prompt
print(type(auto).__name__, "→", [type(s).__name__ for s in auto.steps])
print(auto.invoke("lcel").to_messages())

## Step 02 · RunnablePassthrough
- 입력값이 그대로 전달됨

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 문자열 하나를 받아 프롬프트 변수 두 개에 나눠 넣기
qa_prompt = ChatPromptTemplate.from_template("[{level}] 수준으로 답해줘: {question}")
qa_chain = {"question": RunnablePassthrough(), "level": lambda _: "초급"} | qa_prompt | llm | to_str
print(qa_chain.invoke("API가 뭐예요?"))

### 2-2. RunnablePassthrough.assign
- 기존 입력에 새 키 추가

In [ ]:
summary_chain = ChatPromptTemplate.from_template("다음 글을 한 문장으로 요약해줘:\n{text}") | llm | to_str

enriched = RunnablePassthrough.assign(
    summary=summary_chain,                    # 체인 결과를 새 키로
    length=lambda x: len(x["text"]),          # 함수 결과를 새 키로
)
out = enriched.invoke({"text": "LangChain은 LLM 애플리케이션 개발을 돕는 프레임워크로, 프롬프트 · 모델 · 파서 같은 컴포넌트를 조합해 체인을 만든다."})
print(out.keys())    # text(원본) + summary + length
print(out)

## Step 03 · RunnableParallel 
- 같은 입력으로 여러 체인 동시 실행

In [ ]:
from langchain_core.runnables import RunnableParallel

pros = ChatPromptTemplate.from_template("{topic}의 장점 2가지를 짧게") | llm | to_str
cons = ChatPromptTemplate.from_template("{topic}의 단점 2가지를 짧게") | llm | to_str
quiz = ChatPromptTemplate.from_template("{topic}에 대한 OX 퀴즈 1개") | llm | to_str

parallel = RunnableParallel(pros=pros, cons=cons, quiz=quiz)
result = parallel.invoke({"topic": "재택근무"})
for k, v in result.items():
    print(f"[{k}] {v}\n")

In [ ]:
# 파이프 안의 dict 리터럴은 RunnableParallel로 자동 변환
seq = RunnablePassthrough() | {"pros": pros, "cons": cons}
print([type(s).__name__ for s in seq.steps])

### 3-2. itemgetter 

- 입력 dict에서 필요한 키만 꺼내기

In [ ]:
from operator import itemgetter

translate_prompt = ChatPromptTemplate.from_template("다음 문장을 {lang}로 번역해줘: {sentence}")
translate_chain = (
    {"sentence": itemgetter("text"), "lang": itemgetter("target")}
    | translate_prompt | llm | to_str
)
print(translate_chain.invoke({"text": "오늘 날씨가 좋네요", "target": "영어", "user_id": 123}))

In [ ]:
from langchain_core.prompts import PromptTemplate

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | llm
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | llm
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

combined.invoke({"country": "대한민국"})

## Step 04 · RunnableLambda 

- 파이썬 함수를 체인에 넣기

규칙: 입력은 **하나** (여러 값이 필요하면 dict로 받기)

In [ ]:
import re
from langchain_core.runnables import RunnableLambda


def clean_text(text: str) -> str:
    """전처리: 공백 정리 + 길이 제한"""
    return re.sub(r"\s+", " ", text).strip()[:500]


def to_bullets(text: str) -> list[str]:
    """후처리: 줄 단위 리스트로 변환"""
    return [line.strip("-• ").strip() for line in text.splitlines() if line.strip()]


bullet_prompt = ChatPromptTemplate.from_template("다음 내용의 핵심을 3줄 bullet로 정리해줘:\n{text}")
bullet_chain = (
    RunnableLambda(clean_text)
    | {"text": RunnablePassthrough()}
    | bullet_prompt | llm | to_str
    | RunnableLambda(to_bullets)
)
print(bullet_chain.invoke("   LCEL은   파이프 연산자로   Runnable을   합성한다.\n\n 스트리밍·배치·비동기를 기본 지원한다.  "))

In [ ]:
# 디버그 포인트: 값을 출력하고 그대로 넘기기
def debug(label):
    def _f(x):
        print(f"--- [{label}] {type(x).__name__}: {str(x)[:80]}")
        return x
    return RunnableLambda(_f)


debug_chain = prompt | debug("prompt 출력") | llm | debug("model 출력") | to_str
print(debug_chain.invoke({"topic": "벡터"}))

In [ ]:
# 여러 인자를 받는 함수 → dict 언패킹 래퍼
def discount(price: int, rate: float) -> int:
    return int(price * (1 - rate))


discount_runnable = RunnableLambda(lambda d: discount(**d))
print(discount_runnable.invoke({"price": 10000, "rate": 0.2}))

### 4-2. @chain 데코레이터 

- 함수 안에서 여러 체인을 조합

In [ ]:
from langchain_core.runnables import chain as as_runnable


@as_runnable
def title_and_intro(topic: str) -> dict:
    title = (ChatPromptTemplate.from_template("{t}에 대한 블로그 제목 1개만") | llm | to_str).invoke({"t": topic})
    intro = (ChatPromptTemplate.from_template("제목 '{title}'인 글의 도입부 2문장") | llm | to_str).invoke({"title": title})
    return {"title": title, "intro": intro}


print(type(title_and_intro).__name__)
print(title_and_intro.invoke("LangChain 입문"))

## Step 05 · 분기와 라우팅

### 5-1. RunnableBranch 
- (조건, 체인) 쌍 + 기본 체인

In [ ]:
from langchain_core.runnables import RunnableBranch

code_chain = ChatPromptTemplate.from_template("당신은 시니어 개발자입니다. 코드 예시와 함께 답하세요: {q}") | llm | to_str
biz_chain = ChatPromptTemplate.from_template("당신은 경영 컨설턴트입니다. 비즈니스 관점으로 답하세요: {q}") | llm | to_str
general_chain = ChatPromptTemplate.from_template("친절하게 답하세요: {q}") | llm | to_str

branch = RunnableBranch(
    (lambda x: "코드" in x["q"] or "파이썬" in x["q"], code_chain),
    (lambda x: "매출" in x["q"] or "비용" in x["q"], biz_chain),
    general_chain,   # 기본값
)
for q in ["파이썬으로 리스트 뒤집는 법", "매출을 늘리는 방법", "오늘 기분 좋은 인사말"]:
    print(f"Q: {q}\nA: {branch.invoke({'q': q})[:80]}...\n")

### 5-2. 함수 라우팅
- LLM이 분류하고, 함수가 체인을 선택 (권장)

In [ ]:
classify = (
    ChatPromptTemplate.from_template("다음 질문을 [기술/비즈니스/일반] 중 하나로만 분류해. 한 단어로만 답해.\n질문: {q}")
    | llm | to_str
)


def route(info: dict):
    topic = info["topic"].strip()
    if "기술" in topic:
        return code_chain
    if "비즈니스" in topic:
        return biz_chain
    return general_chain       # Runnable을 반환하면 이어서 실행됨


router_chain = {"topic": classify, "q": itemgetter("q")} | RunnableLambda(route)
print(router_chain.invoke({"q": "REST API와 GraphQL의 차이는?"})[:120])

## Step 06 · 실행 제어

### 6-1. config
- 실행 이름 · 태그 · 메타데이터 · 동시 실행 수 (LangSmith에서 확인)

In [ ]:
topic_chain = prompt | llm | to_str
topics = [{"topic": t} for t in ["RAG", "임베딩", "토큰", "프롬프트"]]
results = topic_chain.batch(
    topics,
    config={"max_concurrency": 2, "run_name": "topic_batch", "tags": ["practice", "lcel"], "metadata": {"step": 6}},
)
print(len(results), results[0][:60])

In [ ]:
# with_config — 체인에 기본 설정을 고정
named_chain = topic_chain.with_config(run_name="one_line_explainer", tags=["explainer"])
print(named_chain.invoke({"topic": "LangSmith"}))

### 6-2. bind 

- 모델 호출 인자 고정

In [ ]:
short_llm = llm.bind(stop=["\n"])             # 첫 줄바꿈에서 생성 중단
print((prompt | short_llm | to_str).invoke({"topic": "LCEL의 장점 여러 가지를 줄바꿈으로"}))

### 6-3. with_retry · with_fallbacks 
- 예외 처리 (재시도 로직, 모델 호출 오류 처리)

In [ ]:
retry_chain = topic_chain.with_retry(stop_after_attempt=3, wait_exponential_jitter=True)
print(retry_chain.invoke({"topic": "재시도"}))

In [ ]:
broken_llm = ChatOpenAI(model="gpt-6-nonexistent", max_retries=0)   # 일부러 실패하는 모델
backup_llm = ChatOpenAI(model=MODEL)
safe_llm = broken_llm.with_fallbacks([backup_llm])
print((prompt | safe_llm | to_str).invoke({"topic": "폴백"}))

### 6-4. configurable_fields 

- 실행 시점에 파라미터 바꾸기

추론 모델은 reasoning effort가 none일 때만 temperature를 지원할 수 있음

In [ ]:
from langchain_core.runnables import ConfigurableField

conf_llm = ChatOpenAI(model=MODEL, reasoning_effort="none", temperature=0).configurable_fields(
    temperature=ConfigurableField(id="temperature", name="Temperature"),
)
name_chain = ChatPromptTemplate.from_template("카페 이름 하나만 지어줘. 이름만 답해.") | conf_llm | to_str
for t in [0.0, 1.5]:
    names = [name_chain.with_config(configurable={"temperature": t}).invoke({}) for _ in range(3)]
    print(t, names)

## Step 07 · 대화 기록 · 디버깅

### 7-1. RunnableWithMessageHistory

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 상담원입니다. 2문장 이내로 답하세요."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
base_chain = chat_prompt | llm | to_str

store: dict[str, InMemoryChatMessageHistory] = {}


def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chat = RunnableWithMessageHistory(
    base_chain, get_history,
    input_messages_key="question",
    history_messages_key="history",
)

cfg_a = {"configurable": {"session_id": "user_a"}}
cfg_b = {"configurable": {"session_id": "user_b"}}
print(chat.invoke({"question": "저는 부산에 가고 싶어요."}, config=cfg_a))
print(chat.invoke({"question": "제가 어디 간다고 했죠?"}, config=cfg_a))
print(chat.invoke({"question": "제가 어디 간다고 했죠?"}, config=cfg_b))   # 다른 세션 → 기억 없음
print({k: len(v.messages) for k, v in store.items()})

### 7-2. 디버깅

In [ ]:
from langchain_core.globals import set_debug

set_debug(True)                    # 모든 Runnable의 입력 · 출력을 상세 출력
(prompt | llm | to_str).invoke({"topic": "디버그"})
set_debug(False)

In [ ]:
# 체인 구조 그래프 출력 (pip install grandalf 필요)
try:
    router_chain.get_graph().print_ascii()
except ImportError:
    print(router_chain.get_graph().to_json()["nodes"][:3])

## Step 08 · 응용 패턴

### 8-1. 순차 체인 — 요약 → 번역 (앞 체인의 출력이 다음 체인의 입력)

In [ ]:
summarize = ChatPromptTemplate.from_template("다음 글을 2문장으로 요약해줘:\n{article}") | llm | to_str
translate = ChatPromptTemplate.from_template("다음 문장을 자연스러운 영어로 번역해줘:\n{summary}") | llm | to_str

pipeline = (
    RunnablePassthrough.assign(summary=summarize)
    | RunnablePassthrough.assign(english=translate)
)

In [ ]:
article = "LangChain은 2022년에 공개된 오픈소스 프레임워크로, 다양한 LLM과 도구를 조합해 애플리케이션을 만들 수 있게 해준다. LCEL을 사용하면 체인을 선언적으로 구성할 수 있다."
res = pipeline.invoke({"article": article})

print("요약:", res["summary"], "\n영문:", res["english"])

### 8-2. 병렬 분석 후 종합 

- 여러 관점을 동시에 분석하고 하나로 합치기

In [ ]:
review = "배송은 이틀 늦었지만 포장이 꼼꼼했고, 제품 품질은 가격 대비 훌륭합니다. 고객센터 연결은 조금 어려웠어요."

analysis = RunnableParallel(
    sentiment=ChatPromptTemplate.from_template("리뷰의 전체 감정을 한 단어로: {review}") | llm | to_str,
    good=ChatPromptTemplate.from_template("리뷰에서 좋은 점만 나열: {review}") | llm | to_str,
    bad=ChatPromptTemplate.from_template("리뷰에서 아쉬운 점만 나열: {review}") | llm | to_str,
)

In [ ]:
report = ChatPromptTemplate.from_template(
    "다음 분석을 종합해 판매자에게 줄 3줄 피드백을 작성해.\n감정: {sentiment}\n장점: {good}\n단점: {bad}"
) | llm | to_str

full_chain = analysis | report
print(full_chain.invoke({"review": review}))

## 기타 LCEL Runnable Components

### 캐싱 (Caching)

- 동일한 작업을 여러 번 수행해야 할 경우 API 호출 횟수를 줄이고 응답 속도를 빠르게 할 수 있음

#### InMemoryCache

In [33]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 모델을 생성합니다.
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template("{country} 에 대해서 200자 내외로 요약해줘")

# 체인을 생성합니다.
chain = prompt | llm

In [ ]:
%%time
response = chain.invoke({"country": "한국"})
print(response.content)

In [ ]:
# %pip install -U langchain-community

In [ ]:
%%time

from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache

# 인메모리 캐시를 사용합니다.
set_llm_cache(InMemoryCache())

# 체인을 실행합니다.
response = chain.invoke({"country": "한국"})
print(response.content)

In [ ]:
%%time
# 체인을 실행합니다.
response = chain.invoke({"country": "한국"})
print(response.content)

### SQLiteCache

In [38]:
import os
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache

# 캐시 디렉토리를 생성합니다.
if not os.path.exists("cache"):
    os.makedirs("cache")

# SQLiteCache를 사용합니다.
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db"))

In [ ]:
%%time
# 체인을 실행합니다.
response = chain.invoke({"country": "한국"})
print(response.content)

### 메모리 (Memory)

- 대화 기록을 저장하는 방법 (history)

### ConversationBufferMemory

- Vulnerability로 인해 라이브러리에서 Deprecated, 이후 제거됨

In [ ]:
# Chain 추가하는 방법
from operator import itemgetter
from langchain_core.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI


# ChatOpenAI 모델을 초기화합니다.
model = ChatOpenAI()

# 대화형 프롬프트를 생성합니다. 이 프롬프트는 시스템 메시지, 이전 대화 내역, 그리고 사용자 입력을 포함합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
# 대화 버퍼 메모리를 생성하고, 메시지 반환 기능을 활성화합니다.
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

In [ ]:
memory.load_memory_variables({})  # 메모리 변수를 빈 딕셔너리로 초기화합니다.

In [57]:
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables)
    | itemgetter("chat_history")  # memory_key 와 동일하게 입력합니다.
)

In [ ]:
runnable.invoke({"input": "hi"})

In [59]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

In [60]:
chain = runnable | prompt | model

In [ ]:
# chain 객체의 invoke 메서드를 사용하여 입력에 대한 응답을 생성합니다.
response = chain.invoke({"input": "만나서 반갑습니다. 제 이름은 제이슨입니다."})
print(response.content)  # 생성된 응답을 출력합니다.

In [ ]:
memory.load_memory_variables({})

In [ ]:
memory.save_context(
    {"human": "만나서 반갑습니다. 제 이름은 제이슨입니다."}, {"ai": response.content}
)

memory.load_memory_variables({})

In [ ]:
# 이름을 기억하고 있는지 추가 질의합니다.
response = chain.invoke({"input": "제 이름이 무엇이었는지 기억하세요?"})
# 답변을 출력합니다.
print(response.content)

#### 대화 내용을 자동으로 저장하는 커스텀 메모리 관리

In [65]:
from operator import itemgetter
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, Runnable
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [66]:
class MyConversationChain(Runnable):

    def __init__(self, llm, prompt, memory, input_key="input"):

        self.prompt = prompt
        self.memory = memory
        self.input_key = input_key

        self.chain = (
            RunnablePassthrough.assign(
                chat_history=RunnableLambda(self.memory.load_memory_variables)
                | itemgetter(memory.memory_key)  # memory_key 와 동일하게 입력합니다.
            )
            | prompt
            | llm
            | StrOutputParser()
        )

    def invoke(self, query, configs=None, **kwargs):
        answer = self.chain.invoke({self.input_key: query})
        self.memory.save_context(inputs={"human": query}, outputs={"ai": answer})
        return answer


#### 메모리 종류

- ConversationBufferMemory(대화 버퍼 메모리): 메시지를 저장한 다음 변수에 메시지를 추출
- ConversationBufferWindowMemory(대화 버퍼 윈도우 메모리): 시간이 지남에 따라 k개만큼만 사용
- ConversationTokenBufferMemory(대화 토큰 버퍼 메모리): 최근 대화 저장 시 토큰 길이를 사용
- ConversationEntityMemory(대화 엔티티 메모리): 특정 엔티티에 대한 주어진 사실을 기억
- ConversationKGMemory(대화 지식그래프 메모리): 지식 그래프를 활용해 서로 다른 개체 간 관계를 이해
- ConversationSummaryMemory(대화 요약 메모리): 대화의 요약을 생성하여 기억
- VectorStoreRetrieverMemory(벡터저장소 검색 메모리): 벡터 스토어에서 가장 눈에 띄는 상위 K개 문서 쿼리

In [70]:
# ChatOpenAI 모델을 초기화합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# 대화형 프롬프트를 생성합니다. 이 프롬프트는 시스템 메시지, 이전 대화 내역, 그리고 사용자 입력을 포함합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

# 대화 버퍼 메모리를 생성하고, 메시지 반환 기능을 활성화합니다.
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

# 요약 메모리로 교체할 경우
# memory = ConversationSummaryMemory(
#     llm=llm, return_messages=True, memory_key="chat_history"
# )

conversation_chain = MyConversationChain(llm, prompt, memory)

In [ ]:
conversation_chain.invoke("안녕하세요? 만나서 반갑습니다. 제 이름은 제이슨 입니다.")

In [ ]:
conversation_chain.invoke("제 이름이 뭐라고요?")

### SQLite에 대화내용 저장

In [78]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

# SQLChatMessageHistory 객체를 생성하고 세션 ID와 데이터베이스 연결 파일을 설정
chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

In [79]:
# 사용자 메시지를 추가합니다.
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 제이슨이야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)
# AI 메시지를 추가합니다.
chat_message_history.add_ai_message("안녕 제이슨, 만나서 반가워. 나도 잘 부탁해!")

In [ ]:
# 채팅 메시지 기록의 메시지들
chat_message_history.messages

In [81]:
# Chain에 적용

In [82]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [83]:
prompt = ChatPromptTemplate.from_messages(
    [
        # 시스템 메시지
        ("system", "You are a helpful assistant."),
        # 대화 기록을 위한 Placeholder
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),  # 질문
    ]
)

# chain 을 생성합니다.
chain = prompt | ChatOpenAI(model_name="gpt-4o") | StrOutputParser()

In [84]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name=user_id,
        session_id=conversation_id,
        connection="sqlite:///sqlite.db",
    )

In [85]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",
        annotation=str,
        name="User ID",
        description="Unique identifier for a user.",
        default="",
        is_shared=True,
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation.",
        default="",
        is_shared=True,
    ),
]

In [86]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_chat_history,  # 대화 기록을 가져오는 함수를 설정합니다.
    input_messages_key="question",  # 입력 메시지의 키를 "question"으로 설정
    history_messages_key="chat_history",  # 대화 기록 메시지의 키를 "history"로 설정
    history_factory_config=config_fields,  # 대화 기록 조회시 참고할 파라미터를 설정합니다.
)

In [87]:
# config 설정
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [ ]:
# 질문과 config 를 전달하여 실행합니다.
chain_with_history.invoke({"question": "안녕 반가워, 내 이름은 제이슨이야"}, config)

In [ ]:
# 후속 질문을 실해합니다.
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

In [ ]:
# config 설정
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation2"}}

# 질문과 config 를 전달하여 실행합니다.
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)